# Data exploration
Quick look at the gait parquet files before building features.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
gaits = ['normal', 'antalgic', 'stiff_legged', 'lurch', 'steppage', 'trendelenburg']
df = pd.concat([pd.read_parquet(f'../data/parquet/{g}.parquet') for g in gaits], ignore_index=True)
df.shape

(340895, 83)

In [3]:
df.iloc[:, :10].head()

,subject_id,gait_label,walk_id,sensor_id,frame_index,source_file,n_frames,time,joint0_x,joint0_y
0,1,normal,1,1,0,human1_normal1/human1_normal1_SkeletonData1.csv,77,12.040,-0.063138,-0.217130
1,1,normal,1,1,1,human1_normal1/human1_normal1_SkeletonData1.csv,77,12.075,-0.041459,-0.186096
2,1,normal,1,1,2,human1_normal1/human1_normal1_SkeletonData1.csv,77,12.107,-0.041850,-0.154736
3,1,normal,1,1,3,human1_normal1/human1_normal1_SkeletonData1.csv,77,12.141,-0.040571,-0.123253
4,1,normal,1,1,4,human1_normal1/human1_normal1_SkeletonData1.csv,77,12.175,-0.045611,-0.095217


First 8 columns are metadata + time, the rest are 25 joints x (x, y, z).

In [4]:
df.dtypes.value_counts()

float32    75
int64       4
str         3
float64     1
Name: count, dtype: int64

### Class balance

In [5]:
df.groupby('gait_label')['source_file'].nunique()

gait_label
antalgic         472
lurch            460
normal           465
steppage         467
stiff_legged     470
trendelenburg    460
Name: source_file, dtype: int64

In [6]:
df.groupby('gait_label')['subject_id'].nunique()

gait_label
antalgic         10
lurch            10
normal           10
steppage         10
stiff_legged     10
trendelenburg    10
Name: subject_id, dtype: int64

Each walk has several sensor files, so one 'sample' later = one sensor file.

In [7]:
df['sensor_id'].value_counts()

sensor_id
3    65410
5    60415
4    57668
6    57605
2    50438
1    49359
Name: count, dtype: int64

In [8]:
df.groupby('source_file')['n_frames'].first().describe()

count    2794.000000
mean      122.009664
std        39.547280
min        52.000000
25%        90.000000
50%       117.000000
75%       145.000000
max       305.000000
Name: n_frames, dtype: float64

### Bad values
The build step warned about huge numbers and some NaNs, checking what is left.

In [9]:
coords = df.filter(regex='^joint')
coords.isna().sum().sum()

np.int64(29)

In [10]:
bad = coords.isna().any(axis=1) | (coords.abs() > 10).any(axis=1)
bad.sum()

np.int64(97)

In [11]:
df[bad]['source_file'].nunique()

65

### A quick look at the signal

In [12]:
df[~bad].groupby('gait_label')['joint14_y'].mean()

gait_label
antalgic         0.641914
lurch            0.624802
normal           0.626882
steppage         0.588701
stiff_legged     0.594803
trendelenburg    0.577188
Name: joint14_y, dtype: float32